In [2]:
#Install sentence-transformers for dense retrieval and reranking, along with Hugging Face transformers for generation.
!pip install -q sentence-transformers transformers torch

In [3]:
# Dense Retrieval (Bi-Encoder Search)
# Extract the top candidate documents from a corpus based on semantic vector similarity.
from sentence_transformers import SentenceTransformer, util

# 1. Initialize Bi-Encoder Model
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Knowledge Base (Document Collection)
corpus = [
    "The Transformer model architecture relies entirely on attention mechanisms and was introduced in 2017.",
    "BERT is a bidirectional representation model trained using Masked Language Modeling.",
    "Generative Pre-trained Transformers (GPT) use decoder-only architectures to generate autoregressive text.",
    "Retrieval-Augmented Generation (RAG) retrieves relevant document chunks to ground LLM responses with accurate external knowledge."
]

# 3. Compute static/contextual document embeddings
corpus_embeddings = bi_encoder.encode(corpus, convert_to_tensor=True)

# 4. User Query
query = "What architecture helps ground LLMs with external document knowledge?"

# 5. Retrieve top hits using Cosine Similarity
query_embedding = bi_encoder.encode(query, convert_to_tensor=True)
search_results = util.semantic_search(query_embedding, corpus_embeddings, top_k=3)[0]

print(f"Query: '{query}'\n")
print("--- DENSE RETRIEVAL CANDIDATES ---")
for hit in search_results:
    idx = hit['corpus_id']
    score = hit['score']
    print(f"Score: {score:.4f} | Document: {corpus[idx]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Query: 'What architecture helps ground LLMs with external document knowledge?'

--- DENSE RETRIEVAL CANDIDATES ---
Score: 0.5723 | Document: Retrieval-Augmented Generation (RAG) retrieves relevant document chunks to ground LLM responses with accurate external knowledge.
Score: 0.1876 | Document: Generative Pre-trained Transformers (GPT) use decoder-only architectures to generate autoregressive text.
Score: 0.1832 | Document: The Transformer model architecture relies entirely on attention mechanisms and was introduced in 2017.


In [4]:
#Candidate Reranking (Cross-Encoder)
#Pass the top candidate results through a Cross-Encoder reranker model to re-score true query relevance.
from sentence_transformers import CrossEncoder

# Load Cross-Encoder (monoBERT style relevance model)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Pair the user query with each candidate document returned from Bi-Encoder
candidate_docs = [corpus[hit['corpus_id']] for hit in search_results]
query_doc_pairs = [[query, doc] for doc in candidate_docs]

# Predict relevance scores (0 to 1 scale)
scores = cross_encoder.predict(query_doc_pairs)

# Rank documents by score
reranked = sorted(zip(scores, candidate_docs), key=lambda x: x[0], reverse=True)

print("--- CROSS-ENCODER RERANKED RESULTS ---")
for score, doc in reranked:
    print(f"Relevance Score: {score:.4f} | Document: {doc}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

--- CROSS-ENCODER RERANKED RESULTS ---
Relevance Score: 3.7297 | Document: Retrieval-Augmented Generation (RAG) retrieves relevant document chunks to ground LLM responses with accurate external knowledge.
Relevance Score: -9.6275 | Document: Generative Pre-trained Transformers (GPT) use decoder-only architectures to generate autoregressive text.
Relevance Score: -10.2440 | Document: The Transformer model architecture relies entirely on attention mechanisms and was introduced in 2017.


In [9]:
# Context-Grounded Generation (Full RAG)

# Pass the top reranked context chunk into an LLM generation pipeline to answer the user query.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load generative synthesis model (Flan-T5)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# Top reranked context chunk
top_context = reranked[0][1]

# Construct RAG Prompt
rag_prompt = f"""Answer the question based ONLY on the context provided below.

Context:
{top_context}

Question:
{query}

Answer:"""

# Encode the input
input_ids = tokenizer(rag_prompt, return_tensors="pt").input_ids

# Generate output
outputs = model.generate(input_ids, max_new_tokens=60, do_sample=False)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- RAG OUTPUT ---")
print("Retrieved Context:\n", top_context)
print("\nGenerated Response:\n", generated_text)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


--- RAG OUTPUT ---
Retrieved Context:
 Retrieval-Augmented Generation (RAG) retrieves relevant document chunks to ground LLM responses with accurate external knowledge.

Generated Response:
 Retrieval-Augmented Generation
